In [2]:
# Use jax to compute
import jax
import jax.numpy as jnp
# using dataclass to store data
from dataclasses import dataclass

# pre-defined orbitals
orbital_list = ['s','p','d','f']

# using double precision
jax.config.update("jax_enable_x64", True)
default_type_complex = jnp.complex128
default_type_real = jnp.float64



# utilty

In [3]:
#utilty ================================ BEGIN

# fermi-dirac funtion
def fermi_dirac(E,mu,kBT) -> float:
    return 1.0/(jnp.exp((E-mu)/kBT)+1)
# vamp it, E is an array, in  shape of (nE,), 
# return an F-D distrubution of a whole range of E
fermi_dirac_vamp = jax.vmap(fermi_dirac,in_axes=(0,None,None))

# calculate difference matrix
# only need to be computed once
def cal_center_matrix(center_list):
    '''
    param center_list: wannier center list, 2D array, shape (nwan,3)
    return: wannier center difference matrix, 3D array, shape (nwan,nwan,3)

    matrix[i,j] = vec[i] - vec[j]
    '''
    # check input
    assert center_list.ndim == 2, "center_list must be a 2D array"  
    assert center_list.shape[1] == 3, "center_list must have shape (nwan, 3)"
    # calculate wannier center difference matrix
    center_matrix = center_list[:, None, :] - center_list[None, :, :]
    return center_matrix

# find EF from .scf.out or .nscf.out file
def find_EF(file_name: str) -> float:
    '''
    param file_name: name of the .scf.out or .nscf.out file
    return: Fermi energy
    '''
    with open(file_name, "r") as f:
        lines = f.readlines()
    for line in lines:
        if "the Fermi energy is" in line:
            EF = float(line.split()[-2])
            break
    return EF

#utilty ================================ END

In [4]:
# test EF
print (find_EF("Cu.scf.out"))


12.43


# data class

In [12]:
# data class========================= BEGIN
@dataclass
class TB_model:
    HR: jnp.ndarray = None # (nR ,nwan,nwan)
    cell_center: jnp.ndarray = None # (nR,3)
    real_lattice: jnp.ndarray = None # (3,3)
    recip_lattice: jnp.ndarray = None # (3,3)
    center_list: jnp.ndarray = None # (nwan,3)
    AA_R: jnp.ndarray = None # (nR,nwan,nwan,3)
    EF: float = None # Fermi energy



    # post init: calculate some useful quantities
    def __post_init__(self):
        self.nwan = self.HR.shape[-1] # number of wannier functions
        self.nR = self.HR.shape[0] # number of R points
        self.center_matrix = cal_center_matrix(self.center_list) # (nwan,nwan,3)

        b1 = self.recip_lattice[0]
        b2 = self.recip_lattice[1]
        b3 = self.recip_lattice[2]
        self.V = jnp.dot(b1,jnp.cross(b2,b3)) # volume of a reciprocal lattice cell
        
        # operator list
        self.O_list_inner: dict = {} # list of operators
        self.O_list_kubo: dict = {} # list of operator names    
        
        # print some info
        print (f"TB_model: nwan={self.nwan}, nR={self.nR}")
    
    def set_EF(self, file_name: str):
        self.EF = find_EF(file_name)
        print (f"EF={self.EF}")

    def add_operator(self, operator: jnp.ndarray, name: str,category: str="kubo"):
        '''
        add an operator to the model, and check its shape
        param O: operator matrix, shape (nwan,nwan)
        param name: name of the operator
        '''
        # check shape
        if operator.shape != (self.nwan,self.nwan):
            raise ValueError(f"operator {name} shape {operator.shape} not match nwan={self.nwan}")        

        if category == "kubo":
            self.O_list_kubo[name] = operator
            print (f"add operator {name} in kubo list")
        elif category == "inner":
            self.O_list_inner[name] = operator
            print (f"add operator {name} in inner list")
        else:
            raise ValueError(f"category {category} not supported")

    def clear_operator(self, category: str="all"):
        '''
        clear all operators in the model
        '''
        if category == "kubo":
            self.O_list_kubo = {}
            print (f"clear all operators in kubo list")
        elif category == "inner":
            self.O_list_inner = {}
            print (f"clear all operators in inner list")
        elif category == "all":
            self.O_list_kubo = {}
            self.O_list_inner = {}
            print (f"clear all operators in kubo list and inner list")
        else:
            raise ValueError(f"category {category} not supported")




In [16]:



model = read_TB_model("TB.npz")

model.add_operator(jnp.eye(model.nwan), "I")

S,L,LS = build_operator_LS(['spd'])
print(L.shape)
model.add_operator(L[:,:,0], "Lz")

O_test = operator_matrix_all(['spd'])
print(O_test.orbital_all)
print(O_test.pre_prepared_operators.keys())
print(O_test.pre_prepared_operators["LS"].shape)
LS = O_test.pre_prepared_operators["LS"]
model.add_operator(LS, "LS")
Lz_p = O_test.make_POP_easy("d","Lz")
model.add_operator(Lz_p, "Lzp")

P_p = O_test.make_P_easy('p')    
model.add_operator(P_p, "Pp",category='inner')

print (model.O_list_kubo.keys())
print (model.O_list_inner.keys())


read_TB_model: read TB.npz
TB_model: nwan=18, nR=617
add operator I in kubo list
(18, 18, 3)
add operator Lz in kubo list
['s', 's', 'p', 'p', 'p', 'p', 'p', 'p', 'd', 'd', 'd', 'd', 'd', 'd', 'd', 'd', 'd', 'd']
dict_keys(['Sx', 'Sy', 'Sz', 'Lx', 'Ly', 'Lz', 'LS', 'I'])
(18, 18)
add operator LS in kubo list
add operator Lzp in kubo list
add operator Pp in inner list
dict_keys(['I', 'Lz', 'LS', 'Lzp'])
dict_keys(['Pp'])


# Calculators

In [ ]:
# Calculators==================Begin
def make_calc_inner_product(
    O_array:jnp.ndarray, # nO * nwan *nwan
):
    pass



#hamiltonian in k space
# use functional factory to make a calculator
def make_k_hamiltonian_calculator(
    cell_center:jnp.ndarray, # nR * 3
    center_matrix:jnp.ndarray, # nwan * nwan * 3
    HR:jnp.ndarray, # nR * nwan * nwan
):

    @jax.jit
    def _calculator(k:jnp.ndarray):
        '''
        calculate the hamiltonian in k space
        para:
            k: jnp.ndarray, # 3
        return:
            Hk: jnp.ndarray, # nwan * nwan
        '''
        # calculate the phase factor for each R
        # This phase is caused by the diffrent position of the cell center
        dot_product = jnp.einsum('ij,j->i', cell_center, k)
        Phase_R = jnp.exp(1j * dot_product) # nR

        # calculate the phase factor for each center diff
        # This phase is caused by the diffrent position of the WF within the unit cell
        dot_product = jnp.einsum('ijk,k->ij', center_matrix, k)
        Phase_W = jnp.exp(1j * dot_product) # nwan * nwan
        
        # part 1: use einsum to calculate the partial sum of Phase_R and HR
        partial_sum = jnp.einsum('i,ijk->jk', Phase_R, HR)
        # part 2: multiply the phase factor for each center diff
        Hk = partial_sum * Phase_W

        # make sure the hamiltonian is hermitian
        return (Hk + Hk.conj().T) / 2

    return _calculator


def make_k_hamiltonian_velocity_calculator(
    cell_center:jnp.ndarray, # nR * 3
    center_matrix:jnp.ndarray, # nwan * nwan * 3
    HR:jnp.ndarray, # nR * nwan * nwan
    AA_R:jnp.ndarray, # nR * nwan * nwan * 3
    use_A:bool = False, # whether to use the Anomalous position term from Go 
    #PhysRevB.109.174435(2024)
):
    nwan = center_matrix.shape[0]

    if use_A:
        print("Anomalous position term is used")
    else:
        print("Anomalous position term is not used,use partial only")

    @jax.jit
    def _calculator(k:jnp.ndarray):
        '''
        calculate the velocity of the hamiltonian with respect to k
        para:
            k: jnp.ndarray, # 3
        return:
            Hk: jnp.ndarray, # nwan * nwan
            velocity_k: jnp.ndarray, # nwan * nwan * 3
        '''
        # calculate 2 phase factors
        dot_product = jnp.einsum('ij,j->i', cell_center, k)
        Phase_R = jnp.exp(1j * dot_product) # nR
        dot_product = jnp.einsum('ijk,k->ij', center_matrix, k)
        Phase_W = jnp.exp(1j * dot_product) # nwan * nwan

        # Hk hamiltonian in k space
        # part 1: use einsum to calculate the partial sum of Phase_R and HR
        partial_sum = jnp.einsum('i,ijk->jk', Phase_R, HR) # nwan * nwan   
        Hk = partial_sum * Phase_W
        Hk = (Hk + Hk.conj().T) / 2

        # There are 2 parts of derivative of Hk with respect to k
        # ∂H(k)/∂k_μ = i Σ_R e^{ik·(R+τ_ij)} (R_μ+τ_ij,μ) H(R)
        part1 = jnp.einsum('iab, i, ab, id -> abd', 
                            HR, 
                            Phase_R, 
                            Phase_W, 
                            cell_center
                            )
        # part 2: 
        part2 = jnp.einsum('iab, i, ab, abd -> abd', 
                            HR, 
                            Phase_R, 
                            Phase_W, 
                            center_matrix)  
        # dH/dk  = i (part1 - part2)
        Hk_derivative = 1j*(part1 - part2)

        if use_A:
            # A_{mn}(k) = -i ( <R|A|R>_{mn} - <R|A|R>_{nm}^* )
            partial_sum = jnp.einsum('i,ijkl->jkl', Phase_R, AA_R)
            temp = (partial_sum + partial_sum.transpose(1, 0, 2).conj()) / 2
            A_term = -1j * (jnp.einsum('ijl,jk->ikl',temp,Hk) - jnp.einsum('ij,jkl->ikl',Hk,temp))
            velocity_k = Hk_derivative + A_term
        else:
            velocity_k = Hk_derivative
        return Hk,velocity_k    

        
# Calculators==================End



# IO functions

In [7]:
# io ================================ BEGIN
def read_TB_model(file_name: str) -> TB_model:
    """
    Read a TB model from a npz file.

    Parameters
    ----------
    file_name : str
        The name of the npz or pickle file.

    Returns
    -------
    TB_model
        The TB model.
    """

    # check file extension
    if file_name.endswith(".npz"):
        print (f"read_TB_model: read {file_name}")
        data = jnp.load(file_name)
        HR = data['HR']
        cell_center = data['cRvec']
        center_list = data['Wan_center']
        AA_R = data['AA_R']
        real_lattice = data['real_lattice']
        recip_lattice = data['recip_lattice']
        
        return TB_model(HR=HR,
                        cell_center=cell_center,
                        center_list=center_list,
                        AA_R=AA_R,
                        real_lattice=real_lattice,
                        recip_lattice=recip_lattice)
    elif file_name.endswith(".pickle"):
        import pickle
        with open(file_name, "rb") as f:
            model = pickle.load(f)
        return model
    else:
        raise ValueError(f"Unknown file extension {file_name}")


def save_TB_model(model: TB_model, file_name: str):
    """
    Save a TB model to a pickle file.

    Parameters
    ----------
    model : TB_model
        The TB model to be saved.
    file_name : str
        The name of the pickle file.
    """
    import pickle
    with open(file_name+".pickle", "wb") as f:
        pickle.dump(model, f)
# io ================================ END

# Operators

In [14]:
num_ao = { # num of atomic orbitals for each orbital type, considering spin
    's':2,
    'p':6,
    'd':10,
    'f':14, # no g orbital
}
# load pre-defined operator matrix
operator = jnp.load("operator.npz")

class operator_matrix_all:
    def __init__(self,orbital_list:list[str]):
        self.orbital_list = orbital_list
        self.orbital_all = build_operator_all(orbital_list)
        self.ndim = len(self.orbital_all)

        self.S,self.L,self.LS = build_operator_LS(orbital_list,use_LS_base=True)

        self.pre_prepared_operators = {
            'Sx' :self.S[:,:,0],
            'Sy' :self.S[:,:,1],
            'Sz' :self.S[:,:,2],
            'Lx' :self.L[:,:,0],
            'Ly' :self.L[:,:,1],
            'Lz' :self.L[:,:,2],
            'LS' :self.LS,
            "I"  :jnp.eye(self.ndim,dtype=default_type_real),
        }


    
    def make_P_easy(self,select_name:str):
        """
        make projection operator for given orbital name

        param select_name: name of the orbital, e.g. "s" or "p"
        return: projection operator, shape (norb,norb)
        """
        return build_P_easy(self.orbital_all,select_name)
    
    def make_POP_easy(self,select_name:str,pre_perpared_name:str):
        """
        make projection operator for given orbital name

        param select_name: name of the orbital, e.g. "s" or "p"
        param pre_perpared_name: name of the pre-prepared operator, e.g. "Sx" or "Ly"
        return: projected operator, P O P ,shape (norb,norb)
        """
        return build_P_easy(self.orbital_all,select_name) @ self.pre_prepared_operators[pre_perpared_name] @ build_P_easy(self.orbital_all,select_name) 

    def make_POP_pre_prepared(self,P_mat:jnp.ndarray,pre_perpared_name:str):
        """
        make projection operator for given orbital name

        param select_name: name of the orbital, e.g. "s" or "p"
        param pre_perpared_name: name of the pre-prepared operator, e.g. "Sx" or "Ly"
        return: projected operator, P O P ,shape (norb,norb)
        """
        return P_mat @ self.pre_prepared_operators[pre_perpared_name] @ P_mat

def build_P_easy(orbital_all:list[str],selected:str):
    """
    build projection operator for given orbital list

    param orbital_list: list of orbital names, e.g. ["spd","p"]
    return: projection operator, shape (norb,norb)
    """
    select_list = [1 if o==selected else 0 for o in orbital_all]    
    return jnp.diag(jnp.array(select_list,dtype=default_type_real))


def build_operator_all(orbital_list:list[str]):
    """
    build operator name list for given orbital list

    param orbital_list: list of orbital names, e.g. ["spd","p"]
    return: operator name list repeat for each orbital type, 
    e.g. ["s","s","p","p","p","d","d","d"]
    """
    orbital_str = "".join(orbital_list)
    orbital_all = []
    for o in orbital_str:
        if o not in num_ao:
            raise ValueError(f"orbital type {o} not supported")
        num_repeat = num_ao[o]
        orbital_all.extend([o]*num_repeat)
 
    return orbital_all

def select_orbital_easy(orbital_name_list:list[str],selected:str):
    """
    select orbital (1 or 0) for given orbital name list

    param orbital_name_list: list of orbital names, e.g. ["s","s","p","p","p","d","d","d"]
    param orbital_name: name of the orbital, e.g. "s" or "p"
    return: same size list of 1 or 0, e.g. [1,1,0,0,0,1,1,1]
    """
    return [1 if o==selected else 0 for o in orbital_name_list]

def build_operator_LS(orbital_list:list[str],use_LS_base:bool=False):
    """
    build operator matrix for given orbital list

    param orbital_list: list of orbital names, e.g. ["spd","p","sd",'f']
    param use_LS_base: whether to use LS coupled base, 
    default False, using real Atomic orbital basis
    return: S,L in 3D array, shape (norb,norb,3) and LS in 2D array, shape (norb,norb)
    """
    orbital_str = "".join(orbital_list)

    orbital_all = []
    for o in orbital_str:
        if o not in num_ao:
            raise ValueError(f"orbital type {o} not supported")
        orbital_all.append(num_ao[o])
    
    # init operator matrix
    norb = sum(orbital_all)
    S = jnp.zeros((norb,norb,3),dtype=default_type_complex)
    L = jnp.zeros((norb,norb,3),dtype=default_type_complex)

    # choose which operator matrix to use
    if use_LS_base:
        S_all = operator["S_LS"]
        L_all = operator["L_LS"]
    else:
        S_all = operator["S"]
        L_all = operator["L"]

    # seperate each orbitals into a dictionary
    S_AO = {
        2:S_all[0:2,0:2,:], # 2
        6:S_all[2:8,2:8,:], # 6
        10:S_all[8:18,8:18,:], # 10
        14:S_all[18:32,18:32,:], # 14    
    }
    L_AO = {
        2:L_all[0:2,0:2,:],# 2
        6:L_all[2:8,2:8,:],# 6
        10:L_all[8:18,8:18,:],# 10  
        14:L_all[18:32,18:32,:],# 14
    }  

    # fill L and S matrix
    i = 0
    for n_ao in orbital_all:
        start = i
        end = i + n_ao
        S = S.at[start:end,start:end,:].set(S_AO[n_ao])
        L = L.at[start:end,start:end,:].set(L_AO[n_ao])
        i = end
    
    # L dot S
    LS = L[:,:,0] @ S[:,:,0] + L[:,:,1] @ S[:,:,1] + L[:,:,2] @ S[:,:,2]

    return S,L,LS


# test
if __name__ == "__main__":
    orbital_list = ["spd"]
    S,L,LS = build_operator_LS(orbital_list,use_LS_base=True)
    test1 = operator_matrix_all(['spd'])
    print(test1.orbital_all)
    print(test1.pre_prepared_operators.keys())
    print(test1.pre_prepared_operators["LS"].shape)

    O_test_P = test1.make_P_easy("s")
    print(jnp.diag(O_test_P))


['s', 's', 'p', 'p', 'p', 'p', 'p', 'p', 'd', 'd', 'd', 'd', 'd', 'd', 'd', 'd', 'd', 'd']
dict_keys(['Sx', 'Sy', 'Sz', 'Lx', 'Ly', 'Lz', 'LS', 'I'])
(18, 18)
[1. 1. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.]


# test

In [23]:
model = read_TB_model("TB.npz")
# test save
save_TB_model(model, "test")

# test load
model2 = read_TB_model("test.pickle")

# test set EF
model2.set_EF("Cu.scf.out")


read_TB_model: read TB.npz
TB_model: nwan=18, nR=617
EF=12.43
